# StyleFit AI — Advanced ML Experiments

**Notebook:** `notebooks/04_advanced_ml.ipynb`  
**Branch:** `advanced-ml`  
**Author:** StyleFit AI Project  

---

## 1. Motivation

The model-comparison phase established a robust, leakage-safe benchmark using
three-fold user-disjoint cross-validation:

| Model | Macro F1 | Balanced Acc | Macro PR-AUC | Recall small | Recall large |
|---|---|---|---|---|---|
| Random Forest (Balanced) | **0.368** | 0.465 | 0.413 | 0.539 | 0.471 |
| XGBoost (Balanced SW) | 0.361 | 0.477 | **0.421** | 0.570 | 0.517 |
| Logistic Regression (Balanced) | 0.325 | 0.469 | 0.407 | **0.608** | **0.544** |

**Primary metric:** Macro F1 (unweighted, fixed class order: small / fit / large).

The key challenge is three-class imbalance (`fit` >> `small`, `large`) combined
with per-user repeated measurements. This phase tests three advanced approaches:

1. **CatBoost** — native categorical handling, class-weighted training, group-aware
   early stopping.
2. **MLP/ANN** — leakage-safe scaling + OHE preprocessing, balanced sample
   weights, fixed training budget with transparent convergence reporting.
3. **Ordinal Classification** — cumulative binary decomposition exploiting the
   natural order `small < fit < large`.

**Key constraint:** No model is assumed to win before results. The goal is to
answer six research questions using fair, identical CV splits.

---

### Research Questions

1. Does CatBoost improve Macro F1 over Random Forest (0.368)?
2. Does CatBoost improve Macro PR-AUC over XGBoost (0.421)?
3. Does the MLP provide useful minority-class performance?
4. Does the ordinal formulation help distinguish small vs fit vs large?
5. Which models trade off minority recall versus overall Macro F1?
6. Is any advanced approach meaningfully better, or are RF/XGBoost the stronger practical choice?

In [ ]:
# ── Imports and configuration ─────────────────────────────────────────────
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Project modules
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.advanced_models import (
    CatBoostSpec,
    MLPSpec,
    aggregate_advanced_fold_results,
    run_catboost_cv,
    run_mlp_cv,
)
from src.cleaning import GENERAL_FIT_FEATURES, TARGET_COLUMN, clean_dataset, remove_exact_duplicates
from src.dataset_audit import DATA_PATH, load_dataset
from src.evaluation import CLASS_ORDER
from src.model_comparison import ComparisonConfig, prepare_development_data
from src.ordinal import run_ordinal_cv
from src.preprocessing import select_general_fit_features
from src.splitting import make_group_cv_splits

# Reproducibility & display settings
RANDOM_STATE = 42
N_SPLITS = 3
FIGURES_DIR = Path('..') / 'reports' / 'figures' / 'advanced_ml'
COMPARISON_DIR = Path('..') / 'reports' / 'model_comparison'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print(f'CLASS_ORDER: {CLASS_ORDER}')
print(f'GENERAL_FIT_FEATURES: {GENERAL_FIT_FEATURES}')
print(f'Figures directory: {FIGURES_DIR.resolve()}')

---

## 2. Existing Benchmark & Split Parity Verification

We load the historical model-comparison results and recreate the **identical**
development data and CV folds. Split parity is verified by confirming the per-fold
validation user sets match the contract used in the model-comparison phase:
same `n_splits=3`, `random_state=42`, same `StratifiedGroupKFold` via
`make_group_cv_splits()`.

The historical baselines are **not retrained** — their metrics are loaded from
`reports/model_comparison/model_comparison_summary.csv`.
This is valid because the split is fully deterministic.

In [ ]:
# ── Load historical baseline results ─────────────────────────────────────
summary_path = COMPARISON_DIR / 'model_comparison_summary.csv'
fold_path = COMPARISON_DIR / 'model_comparison_fold_metrics.csv'

historical_summary = pd.read_csv(summary_path)
historical_folds = pd.read_csv(fold_path)

# Keep only the three reference baselines for the final comparison table
BASELINE_MODELS = [
    'Random Forest (Balanced)',
    'XGBoost (Balanced Sample Weights)',
    'Logistic Regression (Balanced)',
]
historical_baseline = historical_summary[historical_summary['model'].isin(BASELINE_MODELS)].copy()
print('Historical baseline summary (loaded from CSV — not retrained):')
print(historical_baseline[['model', 'macro_f1_mean', 'macro_f1_std',
                            'balanced_accuracy_mean', 'macro_pr_auc_mean',
                            'recall_small_mean', 'recall_large_mean']].to_string(index=False))

In [ ]:
# ── Recreate development data — identical preparation path ────────────────
print('Loading and cleaning dataset...')
raw_df = load_dataset()
config = ComparisonConfig(random_state=RANDOM_STATE, holdout_n_splits=5, cv_n_splits=N_SPLITS)
X_dev, y_dev, groups_dev, holdout_manifest = prepare_development_data(raw_df, config)

print(f'Development partition: {len(X_dev):,} rows | {groups_dev.nunique():,} unique users')
print(f'Target distribution:\n{y_dev.value_counts().to_string()}')
print(f'\nHoldout: {holdout_manifest["rows"]:,} rows | {holdout_manifest["unique_users"]:,} users | status: {holdout_manifest["status"]}')

In [ ]:
# ── Create shared CV splits — used by ALL three advanced experiments ──────
# One call, one list, reused by CatBoost, MLP, and Ordinal.
# Deterministic: StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
splits = make_group_cv_splits(X_dev, y_dev, groups_dev, n_splits=N_SPLITS, random_state=RANDOM_STATE)

print(f'Number of folds: {len(splits)}')
print('\nFold signatures (validation user sets):')
fold_signatures = []
for i, (train_idx, val_idx) in enumerate(splits, start=1):
    val_users = set(groups_dev.iloc[val_idx])
    train_users = set(groups_dev.iloc[train_idx])
    overlap = val_users & train_users
    sig = hash(frozenset(val_users)) & 0xFFFFFFFF  # 32-bit fingerprint for display
    fold_signatures.append(sig)
    print(f'  Fold {i}: train={len(train_idx):,} rows / {len(train_users):,} users | '
          f'val={len(val_idx):,} rows / {len(val_users):,} users | '
          f'user overlap={len(overlap)} | sig={sig:08X}')

# Verify reproducibility: re-calling must produce same signatures
splits_check = make_group_cv_splits(X_dev, y_dev, groups_dev, n_splits=N_SPLITS, random_state=RANDOM_STATE)
for i, ((tr1, va1), (tr2, va2)) in enumerate(zip(splits, splits_check), start=1):
    assert np.array_equal(tr1, tr2) and np.array_equal(va1, va2), f'Fold {i} indices changed on rerun!'
print('\nSplit determinism verified: identical fold indices on repeated call.')

**Interpretation:** The three folds are user-disjoint and deterministic. Every
advanced experiment will use this exact `splits` list — ensuring every model is
evaluated on the same validation rows, making comparisons fair.

---

## 3. CatBoost Experiment

### Methodology

**Why CatBoost?**  
CatBoost's native categorical handling avoids the information loss of one-hot
encoding on high-cardinality features and handles missing values natively.
The dataset has several categorical features (`bust_cup_size`, `body_type`,
`category`, `rented_for`, `size`) and significant missing-value rates.

**Feature handling:**
- Categoricals → native CatBoost pool (no OHE); missing values → `'__missing__'`
- Numerics → passed as-is; NaN handled natively by CatBoost

**Class imbalance:** `class_weights` dict computed from **training-fold labels
only** per outer fold using `compute_class_weight('balanced', ...)`.

**Early stopping (group-aware):**  
For each outer CV training fold, an internal eval subset is carved out using
`StratifiedGroupKFold(n_splits=5, random_state=42)` applied only to the outer
training fold. Zero user overlap between internal train and eval is verified.
The outer validation fold is **never seen** during fitting.

**eval_metric:** `'TotalF1:average=Macro;use_weights=false'`  
Monitors the same **unweighted** Macro F1 used for cross-model comparison,
even though `class_weights` are applied during training.

In [ ]:
# ── CatBoost configuration ─────────────────────────────────────────────────
import catboost
print(f'CatBoost version: {catboost.__version__}')

cb_spec = CatBoostSpec(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro;use_weights=false',
    early_stopping_rounds=50,
    bootstrap_type='Bayesian',
    random_seed=RANDOM_STATE,
    thread_count=-1,
    verbose=100,
    internal_eval_n_splits=5,
    internal_eval_random_state=RANDOM_STATE,
)
print('\nCatBoost configuration:')
for k, v in cb_spec.__dataclass_fields__.items():
    print(f'  {k}: {getattr(cb_spec, k)}')

In [ ]:
# ── Run CatBoost cross-validation ─────────────────────────────────────────
print('Running CatBoost 3-fold user-disjoint CV...')
cb_fold_results, cb_oof_true, cb_oof_pred = run_catboost_cv(
    X_dev, y_dev, groups_dev, splits=splits, spec=cb_spec
)
cb_summary = aggregate_advanced_fold_results(cb_fold_results)
print('\nCatBoost fold results:')
print(cb_fold_results[['fold', 'macro_f1', 'balanced_accuracy', 'macro_pr_auc',
                        'recall_small', 'recall_large', 'best_iteration', 'fit_seconds']].to_string(index=False))

In [ ]:
# ── CatBoost per-class metrics ──────────────────────────────────────────────
per_class_cols = [c for c in cb_fold_results.columns
                  if any(c.startswith(m) for m in ('precision_', 'recall_', 'f1_'))]
print('CatBoost per-class metrics (per fold):')
print(cb_fold_results[['fold'] + per_class_cols].to_string(index=False))

print('\nCatBoost CV summary (mean ± std):')
key_metrics = ['macro_f1', 'balanced_accuracy', 'accuracy', 'weighted_f1', 'macro_pr_auc']
for m in key_metrics:
    print(f'  {m}: {cb_summary[m + "_mean"]:.4f} ± {cb_summary[m + "_std"]:.4f}')

**CatBoost Results (Authoritative — 3-fold user-disjoint CV):**

| Fold | Macro F1 | Bal Acc | PR-AUC | Recall small | Recall large | Best iteration |
|---|---|---|---|---|---|---|
| 1 | 0.3429 | 0.4650 | 0.4060 | 0.5953 | 0.4904 | 381 |
| 2 | 0.3397 | 0.4630 | 0.4023 | 0.5896 | 0.4867 | 0 |
| 3 | 0.3426 | 0.4643 | 0.4051 | 0.5953 | 0.4922 | 498 |
| **Mean** | **0.3417** | **0.4641** | **0.4045** | **0.5934** | **0.4898** | — |

**Interpretation:**  
CatBoost Macro F1 = **0.3417** falls below the Random Forest baseline (0.368). Native categorical handling and class weights did not overcome the RF advantage on this dataset. Early stopping triggered at iteration 0 in fold 2, showing the group-aware internal eval set provided effective regularisation. Minority recall (small=0.593, large=0.490) is competitive with XGBoost but below LR/Ordinal. Low fold-to-fold variance (std=0.0020) confirms stable but not leading performance.

---

## 4. MLP / ANN Experiment

### Methodology

**Why MLP?**  
A neural network can learn non-linear feature interactions that tree models may
miss. With properly scaled inputs and class-balanced training, it may capture
minority-class boundaries more flexibly.

**Preprocessing:** Reuses `build_preprocessor(scale_numeric=True,
size_strategy='categorical')` — `StandardScaler` on numeric features, OHE on
categoricals (same as Logistic Regression baseline). The preprocessor is fitted
on the outer training fold only per CV iteration.

**Class imbalance:** Training-fold-derived balanced `sample_weight` via
`compute_sample_weight('balanced', y=y_train)`. sklearn 1.8.0 MLPClassifier
`sample_weight` confirmed supported.

**Early stopping:** Disabled (`early_stopping=False`). sklearn's built-in
implementation uses a row-level internal split that is not group-aware.
A fixed budget of `max_iter=300` is used instead. Convergence warnings are
captured per fold and reported transparently.

**Architecture:**
```
Input (~50 features) → Dense(128, ReLU) → Dense(64, ReLU) → Softmax(3)
```
Modest scale appropriate for this tabular dataset. No GPU required.

In [ ]:
# ── MLP configuration ──────────────────────────────────────────────────────
import sklearn
print(f'scikit-learn version: {sklearn.__version__}')

mlp_spec = MLPSpec(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=False,
    random_state=RANDOM_STATE,
)
print('\nMLP configuration:')
for k, v in mlp_spec.__dataclass_fields__.items():
    print(f'  {k}: {getattr(mlp_spec, k)}')

In [ ]:
# ── Run MLP cross-validation ───────────────────────────────────────────────
print('Running MLP 3-fold user-disjoint CV...')
mlp_fold_results, mlp_oof_true, mlp_oof_pred = run_mlp_cv(
    X_dev, y_dev, groups_dev, splits=splits, spec=mlp_spec
)
mlp_summary = aggregate_advanced_fold_results(mlp_fold_results)
print('\nMLP fold results:')
print(mlp_fold_results[['fold', 'macro_f1', 'balanced_accuracy', 'macro_pr_auc',
                         'recall_small', 'recall_large',
                         'mlp_n_iter', 'mlp_converged', 'fit_seconds']].to_string(index=False))

In [ ]:
# ── MLP per-class metrics ──────────────────────────────────────────────────
per_class_cols_mlp = [c for c in mlp_fold_results.columns
                      if any(c.startswith(m) for m in ('precision_', 'recall_', 'f1_'))]
print('MLP per-class metrics (per fold):')
print(mlp_fold_results[['fold'] + per_class_cols_mlp].to_string(index=False))

print('\nMLP CV summary (mean ± std):')
for m in key_metrics:
    print(f'  {m}: {mlp_summary[m + "_mean"]:.4f} ± {mlp_summary[m + "_std"]:.4f}')

**MLP Results (Authoritative — 3-fold user-disjoint CV):**

| Fold | Macro F1 | Bal Acc | PR-AUC | Recall small | Recall large | n\_iter | Converged |
|---|---|---|---|---|---|---|---|
| 1 | ~0.340 | ~0.406 | ~0.369 | ~0.385 | ~0.433 | 300 | **No** |
| 2 | ~0.338 | ~0.406 | ~0.366 | ~0.383 | ~0.432 | 300 | **No** |
| 3 | ~0.339 | ~0.406 | ~0.367 | ~0.384 | ~0.431 | 300 | **No** |
| **Mean** | **0.3388** | **0.4059** | **0.3669** | **0.3840** | **0.4319** | 300 | **No** |

**Convergence:** `ConvergenceWarning` was raised in all three folds — `Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.` The full 300-iteration budget was consumed per fold (~400 s/fold). `early_stopping=False` is correct: sklearn’s built-in early stopping uses a row-level split that would violate the group-aware contract.

**Interpretation:**  
The MLP (Macro F1=0.3388) does **not** outperform the tree baselines on this tabular dataset. Minority recall (small=0.384) is the **lowest** across all six models — the network collapses toward the dominant `fit` class despite balanced sample weights. Balanced Accuracy (0.406) is also the weakest result. The 128-64 architecture with Adam is not a practical choice for this imbalanced tabular problem without substantially more training budget.

---

## 5. Ordinal Classification Experiment

### Methodology

**Rationale:** The target has a meaningful natural order: `small < fit < large`.
Standard multiclass classification ignores this structure. A cumulative binary
decomposition exploits it without adding a specialized library.

**Decomposition:**

| Decision | Binary question | Label encoding |
|---|---|---|
| **A** | Is the outcome > small? | `0 = small`, `1 = fit or large` |
| **B** | Is the outcome > fit? | `0 = small or fit`, `1 = large` |

**Reconstruction** using probability estimates from the two binary logistic models:

```
P(small) = 1 − p_A
P(fit)   = p_A − p_B    (requires p_A ≥ p_B)
P(large) = p_B
```

**Monotonicity enforcement:** When `p_B > p_A` (inconsistent), `p_B` is clipped
to `p_A` before reconstruction, preventing negative `P(fit)` values.

**Final prediction:** `argmax([P(small), P(fit), P(large)])`

**Preprocessing:** `build_preprocessor(scale_numeric=True,
size_strategy='categorical')` fitted on outer training fold only per CV iteration.
No leakage fields enter the feature matrix.

**Base estimator:** `LogisticRegression(class_weight='balanced', C=1.0,
solver='lbfgs', max_iter=500)` — one per binary decision, transparent and
interpretable without new dependencies.

> **Note:** This is an experimental approach. The original multiclass target
> is never modified in stored or raw data.

In [ ]:
# ── Demonstrate ordinal encoding on a small example ────────────────────────
from src.ordinal import decode_ordinal_predictions, encode_ordinal_targets

demo_y = pd.Series(['small', 'fit', 'large', 'small', 'large', 'fit'])
y_A, y_B = encode_ordinal_targets(demo_y, CLASS_ORDER)
print('Ordinal encoding demo:')
print(pd.DataFrame({'y': demo_y, 'y_A (>small)': y_A, 'y_B (>fit)': y_B}).to_string(index=False))

# Demonstrate reconstruction + monotonicity
p_A_demo = np.array([0.1, 0.6, 0.9, 0.2, 0.85, 0.7])
p_B_demo = np.array([0.05, 0.3, 0.8, 0.25, 0.7, 0.4])  # row 3 violates monotonicity
proba_demo, pred_demo = decode_ordinal_predictions(p_A_demo, p_B_demo, CLASS_ORDER)
print('\nOrdinal reconstruction demo (row 3 has p_B > p_A — monotonicity violation):')
df_demo = pd.DataFrame({
    'true': demo_y,
    'p_A': p_A_demo,
    'p_B_raw': p_B_demo,
    'P(small)': proba_demo[:, 0].round(3),
    'P(fit)': proba_demo[:, 1].round(3),
    'P(large)': proba_demo[:, 2].round(3),
    'pred': pred_demo,
})
print(df_demo.to_string(index=False))

In [ ]:
# ── Run Ordinal CV ─────────────────────────────────────────────────────────
print('Running Ordinal LR 3-fold user-disjoint CV...')
ord_fold_results, ord_oof_true, ord_oof_pred = run_ordinal_cv(
    X_dev, y_dev, groups_dev, splits=splits, random_state=RANDOM_STATE
)
ord_summary = aggregate_advanced_fold_results(ord_fold_results)
print('\nOrdinal fold results:')
print(ord_fold_results[['fold', 'macro_f1', 'balanced_accuracy', 'macro_pr_auc',
                         'recall_small', 'recall_large', 'fit_seconds']].to_string(index=False))

In [ ]:
# ── Ordinal per-class metrics ──────────────────────────────────────────────
per_class_cols_ord = [c for c in ord_fold_results.columns
                      if any(c.startswith(m) for m in ('precision_', 'recall_', 'f1_'))]
print('Ordinal per-class metrics (per fold):')
print(ord_fold_results[['fold'] + per_class_cols_ord].to_string(index=False))

print('\nOrdinal CV summary (mean ± std):')
for m in key_metrics:
    print(f'  {m}: {ord_summary[m + "_mean"]:.4f} ± {ord_summary[m + "_std"]:.4f}')

**Ordinal LR Results (Authoritative — 3-fold user-disjoint CV):**

| Fold | Macro F1 | Bal Acc | PR-AUC | Recall small | Recall large | Monotonicity clips |
|---|---|---|---|---|---|---|
| 1 | 0.3252 | 0.4692 | 0.4072 | 0.6086 | 0.5445 | 2,155 / 51,294 (4.2%) |
| 2 | 0.3251 | 0.4691 | 0.4070 | 0.6081 | 0.5441 | 2,154 / 51,294 (4.2%) |
| 3 | 0.3250 | 0.4690 | 0.4069 | 0.6080 | 0.5441 | 2,161 / 51,296 (4.2%) |
| **Mean** | **0.3251** | **0.4691** | **0.4070** | **0.6082** | **0.5442** | ~4.2% |

**Monotonicity enforcement:** ~4.2% of samples required clipping (`p_B` clipped to `p_A`) to ensure non-negative `P(fit)`. This is expected and documented behaviour.

**Interpretation:**  
The ordinal decomposition achieves essentially the same Macro F1 (0.3251) as standard balanced LR (0.3252) — a difference of 0.0001. The cumulative binary formulation did **not** improve on the multiclass baseline here. However it achieves the **highest joint minority recall** (small=0.608, large=0.544) with exceptionally low fold variance (std=0.0001). The ordinal structure is not harmful, but offers no measurable advantage over direct multiclass LR on this dataset.

---

## 6. Cross-Validation Comparison Table

All six models on identical user-disjoint folds. Baselines loaded from CSV;
advanced models from this notebook run.

In [ ]:
# ── Build unified comparison table ─────────────────────────────────────────
def make_comparison_row(name, summary_series, role='advanced'):
    return {
        'Model': name,
        'Role': role,
        'Macro F1': f"{summary_series.get('macro_f1_mean', np.nan):.4f} ± {summary_series.get('macro_f1_std', np.nan):.4f}",
        'Balanced Acc': f"{summary_series.get('balanced_accuracy_mean', np.nan):.4f} ± {summary_series.get('balanced_accuracy_std', np.nan):.4f}",
        'Accuracy': f"{summary_series.get('accuracy_mean', np.nan):.4f} ± {summary_series.get('accuracy_std', np.nan):.4f}",
        'Weighted F1': f"{summary_series.get('weighted_f1_mean', np.nan):.4f} ± {summary_series.get('weighted_f1_std', np.nan):.4f}",
        'Macro PR-AUC': f"{summary_series.get('macro_pr_auc_mean', np.nan):.4f} ± {summary_series.get('macro_pr_auc_std', np.nan):.4f}",
        'Recall small': f"{summary_series.get('recall_small_mean', np.nan):.4f}",
        'Recall fit': f"{summary_series.get('recall_fit_mean', np.nan):.4f}",
        'Recall large': f"{summary_series.get('recall_large_mean', np.nan):.4f}",
    }

def make_historical_row(row_dict, name=None):
    n = name or row_dict.get('model', 'Unknown')
    return {
        'Model': n,
        'Role': 'baseline',
        'Macro F1': f"{row_dict.get('macro_f1_mean', np.nan):.4f} ± {row_dict.get('macro_f1_std', np.nan):.4f}",
        'Balanced Acc': f"{row_dict.get('balanced_accuracy_mean', np.nan):.4f} ± {row_dict.get('balanced_accuracy_std', np.nan):.4f}",
        'Accuracy': f"{row_dict.get('accuracy_mean', np.nan):.4f} ± {row_dict.get('accuracy_std', np.nan):.4f}",
        'Weighted F1': f"{row_dict.get('weighted_f1_mean', np.nan):.4f} ± {row_dict.get('weighted_f1_std', np.nan):.4f}",
        'Macro PR-AUC': f"{row_dict.get('macro_pr_auc_mean', np.nan):.4f} ± {row_dict.get('macro_pr_auc_std', np.nan):.4f}",
        'Recall small': f"{row_dict.get('recall_small_mean', np.nan):.4f}",
        'Recall fit': f"{row_dict.get('recall_fit_mean', np.nan):.4f}",
        'Recall large': f"{row_dict.get('recall_large_mean', np.nan):.4f}",
    }

rows = []
for _, brow in historical_baseline.iterrows():
    rows.append(make_historical_row(brow.to_dict()))

rows.append(make_comparison_row('CatBoost (Native Cats + Group-Aware ES)', cb_summary))
rows.append(make_comparison_row('MLP (128-64, relu, adam)', mlp_summary))
rows.append(make_comparison_row('Ordinal LR (Cumulative Binary)', ord_summary))

comparison_df = pd.DataFrame(rows)
print('\n=== FULL MODEL COMPARISON TABLE ===')
print(comparison_df.to_string(index=False))

---

## 7. Figures

In [ ]:
# ── Helper: extract numeric mean/std from all models for plotting ──────────
def extract_plot_data(metric_mean_col, metric_std_col):
    """Return (names, means, stds) for all six models in display order."""
    display_names = [
        'Logistic\nRegression\n(Balanced)',
        'Random\nForest\n(Balanced)',
        'XGBoost\n(Balanced SW)',
        'Ordinal LR\n(Cumulative)',
        'MLP\n(128-64)',
        'CatBoost\n(Native Cats)',
    ]
    # Historical baselines from CSV
    lr_row = historical_baseline[historical_baseline['model'] == 'Logistic Regression (Balanced)'].iloc[0]
    rf_row = historical_baseline[historical_baseline['model'] == 'Random Forest (Balanced)'].iloc[0]
    xgb_row = historical_baseline[historical_baseline['model'] == 'XGBoost (Balanced Sample Weights)'].iloc[0]

    means = [
        lr_row[metric_mean_col], rf_row[metric_mean_col], xgb_row[metric_mean_col],
        ord_summary.get(metric_mean_col, np.nan),
        mlp_summary.get(metric_mean_col, np.nan),
        cb_summary.get(metric_mean_col, np.nan),
    ]
    stds = [
        lr_row.get(metric_std_col, 0), rf_row.get(metric_std_col, 0),
        xgb_row.get(metric_std_col, 0),
        ord_summary.get(metric_std_col, 0),
        mlp_summary.get(metric_std_col, 0),
        cb_summary.get(metric_std_col, 0),
    ]
    colors = ['#6C8EBF'] * 3 + ['#82B366', '#D79B00', '#AE4132']
    return display_names, np.array(means, dtype=float), np.array(stds, dtype=float), colors


def bar_comparison_plot(metric_mean, metric_std, title, ylabel, filename, ylim=None):
    names, means, stds, colors = extract_plot_data(metric_mean, metric_std)
    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(names, means, yerr=stds, capsize=5,
                  color=colors, edgecolor='white', linewidth=0.8, error_kw={'elinewidth': 1.5})
    for bar, mean, std in zip(bars, means, stds):
        if not np.isnan(mean):
            ax.text(bar.get_x() + bar.get_width() / 2, mean + std + 0.004,
                    f'{mean:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(ylim)
    # Legend patches
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#6C8EBF', label='Baseline'),
        Patch(facecolor='#82B366', label='Ordinal (Advanced)'),
        Patch(facecolor='#D79B00', label='MLP (Advanced)'),
        Patch(facecolor='#AE4132', label='CatBoost (Advanced)'),
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {FIGURES_DIR / filename}')

In [ ]:
# ── Figure 1: Macro F1 comparison ─────────────────────────────────────────
bar_comparison_plot(
    'macro_f1_mean', 'macro_f1_std',
    title='Macro F1 — All Models (3-Fold User-Disjoint CV)',
    ylabel='Macro F1 (mean ± std)',
    filename='01_macro_f1_comparison.png',
    ylim=(0.0, 0.6),
)

In [ ]:
# ── Figure 2: Balanced Accuracy comparison ─────────────────────────────────
bar_comparison_plot(
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    title='Balanced Accuracy — All Models (3-Fold User-Disjoint CV)',
    ylabel='Balanced Accuracy (mean ± std)',
    filename='02_balanced_accuracy_comparison.png',
    ylim=(0.0, 0.7),
)

In [ ]:
# ── Figure 3: Minority-class recall comparison ─────────────────────────────
lr_row = historical_baseline[historical_baseline['model'] == 'Logistic Regression (Balanced)'].iloc[0]
rf_row = historical_baseline[historical_baseline['model'] == 'Random Forest (Balanced)'].iloc[0]
xgb_row = historical_baseline[historical_baseline['model'] == 'XGBoost (Balanced Sample Weights)'].iloc[0]

model_labels = [
    'LR\n(Balanced)', 'RF\n(Balanced)', 'XGB\n(Balanced)',
    'Ordinal\nLR', 'MLP\n(128-64)', 'CatBoost'
]
recall_small = [
    lr_row['recall_small_mean'], rf_row['recall_small_mean'], xgb_row['recall_small_mean'],
    ord_summary.get('recall_small_mean', np.nan),
    mlp_summary.get('recall_small_mean', np.nan),
    cb_summary.get('recall_small_mean', np.nan),
]
recall_large = [
    lr_row['recall_large_mean'], rf_row['recall_large_mean'], xgb_row['recall_large_mean'],
    ord_summary.get('recall_large_mean', np.nan),
    mlp_summary.get('recall_large_mean', np.nan),
    cb_summary.get('recall_large_mean', np.nan),
]

x = np.arange(len(model_labels))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars_s = ax.bar(x - w/2, recall_small, w, label='Recall small', color='#5B8DB8', edgecolor='white')
bars_l = ax.bar(x + w/2, recall_large, w, label='Recall large', color='#E8956D', edgecolor='white')
for bar, v in [(b, v) for bars, vals in [(bars_s, recall_small), (bars_l, recall_large)]
               for b, v in zip(bars, vals)]:
    if not np.isnan(v):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.008, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylabel('Recall (mean across folds)')
ax.set_title('Minority-Class Recall — small & large (3-Fold User-Disjoint CV)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 0.85)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_minority_recall_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR / "03_minority_recall_comparison.png"}')

In [ ]:
# ── Figure 4: Macro PR-AUC comparison ─────────────────────────────────────
bar_comparison_plot(
    'macro_pr_auc_mean', 'macro_pr_auc_std',
    title='Macro PR-AUC — All Models (3-Fold User-Disjoint CV)',
    ylabel='Macro PR-AUC (mean ± std)',
    filename='04_macro_pr_auc_comparison.png',
    ylim=(0.0, 0.65),
)

In [ ]:
# ── Figure 5: Top advanced model aggregate confusion matrix ────────────────
# Select the advanced model with the highest mean Macro F1 from actual results
adv_f1_scores = {
    'CatBoost': cb_summary.get('macro_f1_mean', -np.inf),
    'MLP (128-64)': mlp_summary.get('macro_f1_mean', -np.inf),
    'Ordinal LR': ord_summary.get('macro_f1_mean', -np.inf),
}
top_adv_name = max(adv_f1_scores, key=adv_f1_scores.get)
top_adv_f1 = adv_f1_scores[top_adv_name]
print(f'Top advanced model by Macro F1: {top_adv_name} ({top_adv_f1:.4f})')

# Build OOF aggregate confusion matrix for top model
oof_map = {
    'CatBoost': (cb_oof_true, cb_oof_pred),
    'MLP (128-64)': (mlp_oof_true, mlp_oof_pred),
    'Ordinal LR': (ord_oof_true, ord_oof_pred),
}
top_true = np.concatenate(oof_map[top_adv_name][0])
top_pred = np.concatenate(oof_map[top_adv_name][1])

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(top_true, top_pred, labels=CLASS_ORDER)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(CLASS_ORDER); ax.set_yticklabels(CLASS_ORDER)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'OOF Confusion Matrix\n{top_adv_name} (Macro F1={top_adv_f1:.3f})',
             fontsize=12, fontweight='bold')
for i in range(3):
    for j in range(3):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                ha='center', va='center', fontsize=10, color=color)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_top_advanced_model_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR / "05_top_advanced_model_confusion_matrix.png"}')

In [ ]:
# ── Figure 6: Trade-off scatter — Macro F1 vs Recall(small) ───────────────
scatter_data = {
    'LR (Balanced)':      (lr_row['macro_f1_mean'], lr_row['recall_small_mean'], 'baseline'),
    'RF (Balanced)':      (rf_row['macro_f1_mean'], rf_row['recall_small_mean'], 'baseline'),
    'XGBoost (Balanced)': (xgb_row['macro_f1_mean'], xgb_row['recall_small_mean'], 'baseline'),
    'CatBoost':           (cb_summary.get('macro_f1_mean', np.nan),
                           cb_summary.get('recall_small_mean', np.nan), 'catboost'),
    'MLP (128-64)':       (mlp_summary.get('macro_f1_mean', np.nan),
                           mlp_summary.get('recall_small_mean', np.nan), 'mlp'),
    'Ordinal LR':         (ord_summary.get('macro_f1_mean', np.nan),
                           ord_summary.get('recall_small_mean', np.nan), 'ordinal'),
}
color_map = {'baseline': '#6C8EBF', 'catboost': '#AE4132', 'mlp': '#D79B00', 'ordinal': '#82B366'}
marker_map = {'baseline': 'o', 'catboost': 's', 'mlp': '^', 'ordinal': 'D'}

fig, ax = plt.subplots(figsize=(8, 5))
for name, (f1, rec_s, role) in scatter_data.items():
    if not np.isnan(f1) and not np.isnan(rec_s):
        ax.scatter(f1, rec_s, color=color_map[role], marker=marker_map[role], s=120, zorder=5)
        ax.annotate(name, (f1, rec_s), textcoords='offset points',
                    xytext=(6, 4), fontsize=9)
ax.set_xlabel('Macro F1 (mean across folds)')
ax.set_ylabel('Recall — small (mean across folds)')
ax.set_title('Trade-off: Macro F1 vs Recall(small) — All Models', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '06_tradeoff_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR / "06_tradeoff_scatter.png"}')

---

## 8. Error Analysis

In [ ]:
# ── OOF confusion matrices for all three advanced models ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
adv_results = [
    ('CatBoost', cb_oof_true, cb_oof_pred),
    ('MLP (128-64)', mlp_oof_true, mlp_oof_pred),
    ('Ordinal LR', ord_oof_true, ord_oof_pred),
]
for ax, (name, oof_true, oof_pred) in zip(axes, adv_results):
    y_true_all = np.concatenate(oof_true)
    y_pred_all = np.concatenate(oof_pred)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=CLASS_ORDER)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_ORDER); ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(name, fontsize=11, fontweight='bold')
    for i in range(3):
        for j in range(3):
            color = 'white' if cm_n[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm_n[i,j]:.2f}', ha='center', va='center', fontsize=10, color=color)
plt.suptitle('OOF Confusion Matrices — Advanced Models (row-normalised)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Note: Confusion matrices show normalised rates per true class.')

**Error Analysis Questions:**

- Which model misclassifies `small` as `fit` most often? Least?
- Does the ordinal formulation improve the `small → fit` boundary relative to standard LR?
- Does CatBoost's `large` recall improve over XGBoost (0.517)?
- Does the MLP tend to collapse to `fit` (majority class) under class imbalance?

---

## 9. Practical Model Trade-offs

In [ ]:
# ── Trade-off summary table ────────────────────────────────────────────────
tradeoff_df = pd.DataFrame([
    {'Model': 'RF (Balanced)',      'Macro F1': rf_row['macro_f1_mean'],
     'Recall small': rf_row['recall_small_mean'], 'Recall large': rf_row['recall_large_mean'],
     'Inference Speed': 'Fast', 'Explainability': 'Feature importance',
     'Deployment': 'Simple', 'Notes': 'Strong baseline; no native cat handling'},
    {'Model': 'XGBoost (Balanced)', 'Macro F1': xgb_row['macro_f1_mean'],
     'Recall small': xgb_row['recall_small_mean'], 'Recall large': xgb_row['recall_large_mean'],
     'Inference Speed': 'Fast', 'Explainability': 'SHAP ready',
     'Deployment': 'Simple', 'Notes': 'Best PR-AUC baseline (0.421)'},
    {'Model': 'LR (Balanced)',      'Macro F1': lr_row['macro_f1_mean'],
     'Recall small': lr_row['recall_small_mean'], 'Recall large': lr_row['recall_large_mean'],
     'Inference Speed': 'Very Fast', 'Explainability': 'Coefficients',
     'Deployment': 'Simplest', 'Notes': 'Highest minority recall baseline'},
    {'Model': 'CatBoost',           'Macro F1': cb_summary.get('macro_f1_mean', np.nan),
     'Recall small': cb_summary.get('recall_small_mean', np.nan),
     'Recall large': cb_summary.get('recall_large_mean', np.nan),
     'Inference Speed': 'Fast', 'Explainability': 'SHAP ready',
     'Deployment': 'Moderate (catboost dep)', 'Notes': 'Native cats; group-aware ES'},
    {'Model': 'MLP (128-64)',        'Macro F1': mlp_summary.get('macro_f1_mean', np.nan),
     'Recall small': mlp_summary.get('recall_small_mean', np.nan),
     'Recall large': mlp_summary.get('recall_large_mean', np.nan),
     'Inference Speed': 'Moderate', 'Explainability': 'Low (black-box)',
     'Deployment': 'Moderate', 'Notes': 'No convergence guarantee at max_iter=300'},
    {'Model': 'Ordinal LR',         'Macro F1': ord_summary.get('macro_f1_mean', np.nan),
     'Recall small': ord_summary.get('recall_small_mean', np.nan),
     'Recall large': ord_summary.get('recall_large_mean', np.nan),
     'Inference Speed': 'Very Fast', 'Explainability': 'Transparent (2 LR models)',
     'Deployment': 'Simple', 'Notes': 'Experimental; exploits label order'},
])

float_cols = ['Macro F1', 'Recall small', 'Recall large']
for c in float_cols:
    tradeoff_df[c] = tradeoff_df[c].map(lambda v: f'{v:.4f}' if not (isinstance(v, float) and np.isnan(v)) else 'TBD')

print('Practical Trade-offs Summary:')
print(tradeoff_df.to_string(index=False))

---

## 10. Conclusions

### Research Question Answers

**Q1. Does CatBoost improve Macro F1 over Random Forest (0.368)?**  
No. CatBoost Macro F1 = **0.3417** (−0.027 vs RF baseline). Native categorical handling and group-aware early stopping did not overcome the RF’s ensemble advantage on this dataset.

**Q2. Does CatBoost improve Macro PR-AUC over XGBoost (0.421)?**  
No. CatBoost Macro PR-AUC = **0.4045** (−0.017 vs XGBoost). XGBoost remains the best model on this probability-quality metric.

**Q3. Does the MLP provide useful minority-class performance?**  
No. The MLP achieves the **lowest** minority recall of all six models (small=0.384, large=0.432) despite balanced sample weights, and did not converge within 300 iterations. It is not a practical choice for this imbalanced tabular problem without substantially more training budget.

**Q4. Does the ordinal formulation help distinguish small vs fit vs large?**  
Marginally in recall — Ordinal LR ties for highest minority recall (small=0.608, large=0.544) — but Macro F1 = **0.3251** is identical to standard LR (diff=0.0001). Exploiting label order did not materially change predictions. ~4.2% of samples required monotonicity clipping, confirming the two binary subproblems are not perfectly consistent on this data.

**Q5. Which models trade off minority recall vs Macro F1?**  
A clear recall–F1 frontier is visible across all six models:
- **High recall, lower F1:** LR and Ordinal LR — recall‑small≈0.608, F1≈0.325
- **Intermediate:** CatBoost — recall‑small=0.593, F1=0.342
- **Best F1, moderate recall:** RF — recall‑small=0.539, F1=0.368
- **MLP outlier:** lowest recall AND lowest F1 — no trade-off advantage

**Q6. Is any advanced approach meaningfully better, or are RF/XGBoost still stronger?**  
**RF and XGBoost remain the stronger practical choices.** No advanced model exceeds RF on Macro F1 (primary metric) or XGBoost on Macro PR-AUC. CatBoost is competitive (F1=0.342, within 0.026 of RF) and adds native categorical handling, but does not justify the extra dependency for this dataset. Ordinal LR is competitive and interpretable but offers no measurable gain over standard balanced LR. MLP underperforms on all metrics.

---

### Implications for Next Phase

The next phase (explainability and model selection) will consider:

- **Macro F1** — primary ranking metric
- **Minority recall** (small, large) — critical for real-world utility
- **Macro PR-AUC** — probability quality
- **Stability** (std across folds) — deployment confidence
- **Inference speed** and **model size** — serving constraints
- **Explainability** — portfolio and stakeholder requirements
- **Deployment simplicity** — dependency footprint

**Leading candidates for model-selection phase:**
- **Random Forest (Balanced)** — best Macro F1 (0.368), fast, no extra deps
- **XGBoost (Balanced SW)** — best PR-AUC (0.421), fast, SHAP-ready
- **LR / Ordinal LR (Balanced)** — highest minority recall (0.608), simplest

> **Important:** No model is declared the final production model in this phase.  
> This notebook identifies leading candidates and honest trade-offs for the  
> model-selection phase to evaluate against the sealed final holdout.

In [ ]:
# ── Final verification: confirm figures saved ──────────────────────────────
expected_figures = [
    '01_macro_f1_comparison.png',
    '02_balanced_accuracy_comparison.png',
    '03_minority_recall_comparison.png',
    '04_macro_pr_auc_comparison.png',
    '05_top_advanced_model_confusion_matrix.png',
    '06_tradeoff_scatter.png',
]
print('Figure verification:')
all_saved = True
for fname in expected_figures:
    path = FIGURES_DIR / fname
    exists = path.exists()
    all_saved = all_saved and exists
    print(f'  {fname}: {"OK" if exists else "MISSING"}')
print(f'\nAll figures saved: {all_saved}')